This notebook documents the process of saving the AlphaEarth embeddings to Google Drive as a GeoTIFF. I then downloaded it locally, or one can use `rclone` to get it to CyVerse.

In [ ]:
import ee

In [7]:

ee.Authenticate()

True

In [9]:
ee.Initialize(project="embeddings-test-496221")

In [13]:
# Bounding box: Willamette National Forest / Cascades study area (Oregon)
aoi = ee.Geometry.BBox(
    west=-122.25240108713064,
    south=44.19136542920598,
    east=-122.07249996408376,
    north=44.28532650370888,
)

print("AOI area (km²):", aoi.area(maxError=1).divide(1e6).getInfo())

AOI area (km²): 149.73898314571272


In [29]:
# AlphaEarth embedding ImageCollection
# Verify the asset path matches what you have access to — common paths:
#   'projects/alpha-earth/embeddings/v1'
#   'projects/sat-io/open-datasets/alpha-earth/...'
ALPHA_EARTH_ASSET = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'

embeddings = ee.ImageCollection(ALPHA_EARTH_ASSET) \
    .filterDate('2022-01-01', '2023-07-31') \
    .filterBounds(aoi) \
    .mosaic() \
    .clip(aoi)


print("Embedding image bands:", embeddings.bandNames().getInfo())

Embedding image bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']


In [34]:
task = ee.batch.Export.image.toDrive(
    image=embeddings,
    description='hja_alpha_earth',
    folder='alpha_earth',
    fileNamePrefix='hja_all_2022',
    region=aoi,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF',
)
task.start()
print("Export task submitted:", task.id)

Export task submitted: 4SKCKE6O4NVUCJ2H3B35M4RL


Can monitor tasks at https://code.earthengine.google.com/tasks